## 1. Carregamento dos dados

In [0]:
from pyspark.sql import functions as F
from itertools import combinations

# Usados apenas para montar e desenhar a matriz de correlação (18x18) no final
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Lê o dataset como um DataFrame Spark (distribuído)
df = spark.read.table('workspace.default.manufacturing_defect_dataset').cache()

print(f'Total de linhas carregadas: {df.count()}')
df.printSchema()

## 2. Inspeção inicial

In [0]:
# Exibe as primeiras linhas
display(df.limit(20))

# Verifica valores nulos por coluna
df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).show()

# Verifica linhas duplicadas
total_linhas = df.count()
duplicadas = total_linhas - df.distinct().count()
print(f'Total de linhas duplicadas: {duplicadas}')

# Exibe o resumo estatístico
df.describe().show()

# Verifica valores negativos nas colunas de horas de manutenção,
# volume de produção e custo de produção
df.filter((F.col('MaintenanceHours') < 0) |
          (F.col('ProductionVolume') < 0) |
          (F.col('ProductionCost') < 0)).select(
              'MaintenanceHours', 'ProductionVolume', 'ProductionCost'
          ).show()

## 3. Validação de regras de negócio e limites físicos

In [0]:
print('*** VALIDAÇÃO DE REGRAS DE NEGÓCIO ***')

# Verifica percentual de inatividade maior que 100%
downtime_invalido = df.filter(F.col('DowntimePercentage') > 100)
print(f'Quantidade de linhas com Downtime maior que 100%: {downtime_invalido.count()}')

# Verifica eficiência energética menor ou igual a zero
eficiencia_invalida = df.filter(F.col('EnergyEfficiency') <= 0)
print(f'Quantidade de linhas com Eficiência Energética menor ou igual a zero: {eficiencia_invalida.count()}')

## 4. Detecção de outliers no custo unitário

Calcula o custo unitário e usa o intervalo interquartil (IQR) para identificar e filtrar outliers.

In [0]:
# Cria uma coluna para o custo unitário
df = df.withColumn('UnitCost', F.col('ProductionCost') / F.col('ProductionVolume'))

# Calcula os quartis e o intervalo interquartil
# (percentile exato, com a mesma interpolação linear do pandas)
quartis = df.agg(
    F.expr('percentile(UnitCost, 0.25)').alias('q1'),
    F.expr('percentile(UnitCost, 0.75)').alias('q3')
).collect()[0]
q1 = quartis['q1']
q3 = quartis['q3']
iqr = q3 - q1

# Define os limites inferior e superior para outliers
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Filtra os outliers de custo unitário
outliers_custo = df.filter((F.col('UnitCost') < limite_inferior) |
                           (F.col('UnitCost') > limite_superior))
print(f'Quantidade de lotes com anomalias de custo unitário: {outliers_custo.count()}')

# Mantém apenas os lotes dentro do limite saudável de custo
df_clean = df.filter(F.col('UnitCost').between(limite_inferior, limite_superior))

print(f'Total de linhas na base original: {df.count()}')
print(f'Total de linhas na base pós tratamento de custos: {df_clean.count()}')

In [0]:
# Seleciona as colunas numéricas (as 17 originais + UnitCost)
colunas_numericas = [f.name for f in df.schema.fields
                     if f.dataType.simpleString() in ('int', 'bigint', 'long', 'double', 'float')]

# Calcula todas as correlações de pares em uma única agregação no Spark
correlacoes = df.agg(*[F.corr(a, b).alias(f'{a}~{b}')
                       for a, b in combinations(colunas_numericas, 2)]).collect()[0].asDict()

# Monta a matriz (18x18) no driver apenas para visualização
corr_matrix = pd.DataFrame(np.eye(len(colunas_numericas)),
                           index=colunas_numericas, columns=colunas_numericas)
for par, valor in correlacoes.items():
    a, b = par.split('~')
    corr_matrix.loc[a, b] = valor
    corr_matrix.loc[b, a] = valor

# Configurar o tamanho da figura
plt.figure(figsize=(14, 10))

# Criar o mapa de calor (heatmap)
sns.heatmap(
    corr_matrix.astype(float), 
    annot=True, 
    cmap='coolwarm', 
    fmt='.2f', 
    linewidths=0.5, 
    vmin=-1, 
    vmax=1
)

plt.title('Matriz de Correlação das Variáveis', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()